In [1]:
import numpy as np

# ==========================================
# 1. ENVIRONMENT SETUP & DEFINITIONS
# ==========================================
JOB_DURATION = 2
PENALTY_PER_JOB = -1.0 / JOB_DURATION  # -0.5 penalty per job per step
DISCOUNT_FACTOR = 0.9
THETA = 1e-6  # Convergence threshold for policy evaluation

# State & Action indices
NUM_STATES = 4
NUM_ACTIONS = 2

FREE_NO_JOB = 0
FREE_JOB_WAITING = 1
BUSY_NO_JOB = 2
BUSY_JOB_WAITING = 3

WAIT = 0
START = 1

state_names = {
    FREE_NO_JOB: "FREE_NO_JOB",
    FREE_JOB_WAITING: "FREE_JOB_WAITING",
    BUSY_NO_JOB: "BUSY_NO_JOB",
    BUSY_JOB_WAITING: "BUSY_JOB_WAITING"
}
action_names = {WAIT: "WAIT", START: "START"}

# 2. Corrected Transition Matrices (P[action, current_state, next_state])
transition_matrix_wait = np.array([
    [0.50, 0.50, 0.00, 0.00],
    [0.00, 1.00, 0.00, 0.00],
    [0.25, 0.25, 0.25, 0.25],
    [0.00, 0.50, 0.00, 0.50],
])

transition_matrix_start = np.array([
    [0.50, 0.50, 0.00, 0.00],
    [0.00, 0.00, 0.50, 0.50],  # Fixed row: CPU becomes guaranteed BUSY
    [0.25, 0.25, 0.25, 0.25],
    [0.00, 0.50, 0.00, 0.50],
])

P = np.array([transition_matrix_wait, transition_matrix_start])

# 3. Reward Matrix Calculation
r = np.zeros((NUM_STATES, NUM_ACTIONS))
jobs_per_state = {
    FREE_NO_JOB: 0,
    FREE_JOB_WAITING: 1,
    BUSY_NO_JOB: 1,
    BUSY_JOB_WAITING: 2,
}

for state in range(NUM_STATES):
    for action in range(NUM_ACTIONS):
        r[state, action] = jobs_per_state[state] * PENALTY_PER_JOB

# ==========================================
# 2. POLICY ITERATION ALGORITHM
# ==========================================

def policy_evaluation(policy, V):
    """Evaluates the current policy until the value function converges."""
    while True:
        delta = 0
        for s in range(NUM_STATES):
            v_old = V[s]
            a = policy[s]
            # Bellman Expectation Equation: V(s) = R(s,a) + gamma * sum( P(s'|s,a) * V(s') )
            V[s] = r[s, a] + DISCOUNT_FACTOR * np.dot(P[a, s], V)
            delta = max(delta, abs(v_old - V[s]))
        if delta < THETA:
            break
    return V

def policy_improvement(policy, V):
    """Improves the policy greedily based on the calculated Value function."""
    policy_stable = True
    for s in range(NUM_STATES):
        old_action = policy[s]
        
        # Calculate action values Q(s, a) for all actions
        action_values = np.zeros(NUM_ACTIONS)
        for a in range(NUM_ACTIONS):
            action_values[a] = r[s, a] + DISCOUNT_FACTOR * np.dot(P[a, s], V)
        
        # Greedy choice
        new_action = np.argmax(action_values)
        policy[s] = new_action
        
        if old_action != new_action:
            policy_stable = False
            
    return policy, policy_stable

def policy_iteration():
    """Main Policy Iteration Loop."""
    # Initialize arbitrary policy (e.g., always WAIT) and zero value function
    policy = np.zeros(NUM_STATES, dtype=int)
    V = np.zeros(NUM_STATES)
    
    iteration = 0
    stable = False
    
    print("--- Starting Policy Iteration ---")
    while not stable:
        iteration += 1
        V = policy_evaluation(policy, V)
        policy, stable = policy_improvement(policy, V)
        print(f"Iteration {iteration}: Current Policy -> {[action_names[a] for a in policy]}")
        
    return policy, V

# Run the solver
optimal_policy, optimal_values = policy_iteration()

# ==========================================
# 3. PRINT RESULTS
# ==========================================
print("\n=== Final Results ===")
print(f"{'State':<20} | {'Optimal Action':<15} | {'State Value':<12}")
print("-" * 55)
for s in range(NUM_STATES):
    print(f"{state_names[s]:<20} | {action_names[optimal_policy[s]]:<15} | {optimal_values[s]:.4f}")


--- Starting Policy Iteration ---
Iteration 1: Current Policy -> ['WAIT', 'WAIT', 'WAIT', 'WAIT']

=== Final Results ===
State                | Optimal Action  | State Value 
-------------------------------------------------------
FREE_NO_JOB          | WAIT            | -4.0909
FREE_JOB_WAITING     | WAIT            | -5.0000
BUSY_NO_JOB          | WAIT            | -5.0000
BUSY_JOB_WAITING     | WAIT            | -5.9091
